In [0]:
import copy
import json
import os
import pandas as pd
import z_schemas as z_s

# JSON Functions

def readStateJson(pipeline_json):
    with open(pipeline_json, mode="r") as file:
        state = json.load(file)
    copy_state = copy.deepcopy(state)
    return state, copy_state

def updateStateJson(updated_state, pipeline_json):
    temp_file = pipeline_json.replace(".json", ".tmp")
    with open(temp_file, mode="w") as file:
        json.dump(updated_state, file, indent=4)
    os.replace(temp_file, pipeline_json)

# Test Functions

def check_field_presence(df, filename):
    print("Running the 'check_field_presence' test...")
    schema = z_s.SCHEMAS[filename]["fields"]
    schema_fields = list(schema.keys())
    if "data_date" in schema_fields:
        schema_fields.remove("data_date")
    file_fields = df.columns.tolist()
    # The missing_fields list is designed to start off as a copy of the schema_fields list
    # As the code progresses, items are removed from this list and the list should ultimately end up empty
    # If it doesn't, this means a field is "missing" from the new file that occurs in the database table
    missing_fields = copy.deepcopy(schema_fields)
    new_fields = []
    for field in file_fields:
        if field in schema_fields:
            missing_fields.remove(field)
        else:
            new_fields.append(field)
    if (not missing_fields) and (not new_fields):
        print("Test passed.")
        return True
    else:
        print(f"The fields of the new file are different than the schema of the database table.")
        if missing_fields:
            print(f"The following fields are missing: {missing_fields}.")
        if new_fields:
            print(f"The following fields are new: {new_fields}.")
        return False

def check_field_uniqueness(df, filename, field):
    print("Running the 'check_field_uniqueness' test...")
    dup_count = df.duplicated(subset=[field]).sum()
    if dup_count == 0:
        print("Test passed.")
        return True
    else:
        print(f"The field '{field}' contains {dup_count} duplicate value(s).")
        return False

# 'filename' is unused in the following function, but I'm keeping it for signature consistency
def check_no_fully_null_columns(df, filename):
    print("Running the 'check_no_fully_null_column' test..")
    fully_null_columns = df.columns[df.isnull().all()].tolist()
    if fully_null_columns:
        print(f"Fully-null columns detected: {fully_null_columns}")
        return False
    print("Test passed.")
    return True

def check_dtype_consistency(df, filename):
    print("Running the 'check_dtype_consistency' test...")
    schema = z_s.SCHEMAS[filename]["fields"]
    for field, dtype in schema.items():
        try:
            if field == "data_date":
                pass
            elif dtype == "TEXT":
                pass
            elif dtype == "DATE":
                pd.to_datetime(df[field], errors="raise", format="mixed")
            elif dtype == "INTEGER":
                pd.to_numeric(df[field], errors="raise", downcast="integer")
        except (ValueError, TypeError) as e:
            print(f"Type check failed for {field} (expected {dtype}): {e}")
            return False
    print("Test passed.")
    return True
